# 예제 05. LSTM 금융 시계열 예측 체험
### AI가 금융 시계열을 예측하는 방법 체험 · 1일 차 · 1.5시간

> **실행 방법**: 위에서부터 순서대로 실행하세요 (`Shift + Enter`).

| 항목 | 내용 |
|---|---|
| 데이터 | 예제 04의 한국은행 예금은행 수신금리 월별 데이터 (2016년 1월 ~ 2026년 6월) |
| 모델 | 강사가 미리 학습한 LSTM 모델 (2016년 1월 ~ 2024년 12월로 학습) |
| 오늘의 결과물 | LSTM 예측값 · 실제값 · 기준모델 비교 그래프 |

**오늘 해결할 질문**: "AI는 다음 달 금리를 맞힐 수 있을까? 틀린다면 왜 틀릴까?"

> 교육생은 모델을 직접 학습하지 않습니다. 이미 학습된 모델을 **불러와 써 보고, 결과를 비판적으로 읽는 것**이 오늘의 목표입니다.

## 0. 실습 준비

이번에는 파일이 **3개**입니다. 아래 셀을 실행하면 실습 데이터를 **GitHub에서 자동으로 내려받습니다.** (강의장 인터넷이 막혀 **[파일 선택]** 버튼이 나오면, 강사가 배포한 파일을 고르면 됩니다.)

- `deposit_rate_monthly.csv` — 금리 데이터
- `lstm_config.json` — 모델 설정 (입력 길이, 0~1 변환 기준값)
- `lstm_deposit_rate.weights.h5` — 강사가 학습한 모델

`데이터·모델 준비 완료: True`가 나오면 성공입니다.

In [ ]:
import os
import urllib.request
import json
import numpy as np
import pandas as pd
import plotly.express as px

DATA_URL = "https://raw.githubusercontent.com/jetsonai/AIFinancialDataAnalysis/main/Day1/data/"
FILES = ["deposit_rate_monthly.csv", "lstm_config.json", "lstm_deposit_rate.weights.h5"]
for f in FILES:                       # 없는 파일은 GitHub에서 내려받습니다
    if not os.path.exists(f):
        try:
            urllib.request.urlretrieve(DATA_URL + f, f)
        except Exception:             # 인터넷이 막혔을 때만 직접 올리기
            from google.colab import files
            files.upload()
            break
print("데이터·모델 준비 완료:", all(os.path.exists(f) for f in FILES))

## 05-1. LSTM은 어떻게 예측할까?

**LSTM**(Long Short-Term Memory)은 숫자를 **순서대로 읽으면서 중요한 흐름을 기억**하는 AI 모델입니다.

> 🧠 **비유** — 최근 12개월 금리 그래프를 한 칸씩 짚어 가며 읽은 사람이 "요즘 계속 내려가는 중이니 다음 달도 조금 내려가겠네"라고 짐작하는 것과 같습니다.
> LSTM은 이 "짐작하는 요령"을 과거 데이터에서 배워 둔 것입니다.

```
[ 직전 12개월 금리 ]  →  [ LSTM : 흐름 기억 ]  →  [ 다음 달 금리 예측 ]
 2024-01 … 2024-12                                   2025-01
```

- **학습**: 2016년 1월 ~ 2024년 12월의 "12개월 → 다음 달" 묶음 96개로 강사가 미리 학습했습니다.
- **시험**: 2025년 1월 ~ 2026년 6월(18개월)은 학습에 **쓰지 않았습니다.** 모델이 처음 보는 기간에서 얼마나 맞히는지 확인합니다.

## 05-2. 금리 데이터와 모델 설정 읽기

In [ ]:
df = pd.read_csv("deposit_rate_monthly.csv", parse_dates=["date"])
금리 = df["deposit_rate_pct"].to_numpy(dtype="float32")
월 = df["date"]

with open("lstm_config.json", encoding="utf-8") as f:
    설정 = json.load(f)
설정

## 05-3. 0~1로 바꾸기 (스케일링)

AI 모델은 0~1 사이의 숫자를 다룰 때 가장 안정적으로 배웁니다.
그래서 금리를 학습 기간의 최저(0.81%) = 0, 최고(4.29%) = 1이 되도록 바꿔서 넣고, 결과는 다시 금리로 되돌립니다.

In [ ]:
def 축소(x):                 # 금리 → 0~1
    return (x - 설정["min"]) / (설정["max"] - 설정["min"])

def 복원(z):                 # 0~1 → 금리
    return z * (설정["max"] - 설정["min"]) + 설정["min"]

print("금리 2.50% →", round(float(축소(2.50)), 3))
print("0.486 →", round(float(복원(0.486)), 2), "%")

## 05-4. "12개월 → 다음 달" 묶음 만들기

모델은 **직전 12개월**을 한 묶음으로 받아 **다음 달 하나**를 예측합니다. 전체 데이터에서 이런 묶음을 한 달씩 옮겨 가며 만듭니다.

In [ ]:
W = 설정["window"]                                                    # 12개월
대상 = np.arange(W, len(금리))                                         # 예측할 달의 위치
X = np.array([축소(금리[i - W:i]) for i in 대상])[..., None]            # 각 달의 직전 12개월
print("묶음 개수:", len(X), "/ 묶음 하나의 모양:", X[0].shape)

In [ ]:
# 첫 번째 묶음 예시: 2016년 1월~12월 금리 → 2017년 1월을 맞혀야 함
예시 = pd.DataFrame({"월": 월.iloc[0:W].dt.strftime("%Y-%m"), "금리": 금리[0:W]})
print("맞힐 달:", 월.iloc[W].strftime("%Y-%m"), "/ 실제 금리:", 금리[W])
예시.T

## 05-5. 강사가 학습한 LSTM 모델 불러오기

모델의 **모양**(LSTM 층 + 출력 층)을 만들고, 강사가 학습해 둔 **가중치**(배운 요령)를 불러와 채웁니다.

In [ ]:
import keras

model = keras.Sequential([
    keras.Input(shape=(W, 1)),           # 입력: 12개월 × 금리 1개
    keras.layers.LSTM(설정["units"]),     # LSTM 층: 흐름을 기억
    keras.layers.Dense(1),               # 출력 층: 다음 달 금리 1개
])
model.load_weights("lstm_deposit_rate.weights.h5")
print("강사가 학습한 모델을 불러왔습니다.")
model.summary()

## 05-6. 처음 보는 기간(2025년 1월 ~ 2026년 6월) 예측하기

학습에 쓰지 않은 18개월에 대해, 매달 **실제 직전 12개월**을 넣고 다음 달을 예측합니다.

In [ ]:
시험 = (월.iloc[대상] >= "2025-01-01").to_numpy()
예측 = 복원(model.predict(X[시험], verbose=0).ravel())

결과 = pd.DataFrame({
    "월": 월.iloc[대상[시험]].values,
    "실제": 금리[대상[시험]].round(2),
    "LSTM 예측": 예측.round(2),
})
결과.head()

## 05-7. 비교할 기준모델 만들기

AI가 잘했는지 알려면 **아주 단순한 방법**과 비교해야 합니다.

- **기준모델 ① 전월값**: "다음 달도 이번 달과 같다"
- **기준모델 ② 3개월 평균**: "다음 달은 최근 3개월 평균이다"

In [ ]:
결과["전월값"] = 금리[대상[시험] - 1].round(2)
결과["3개월평균"] = np.array([금리[i - 3:i].mean() for i in 대상[시험]]).round(2)
결과

## 05-8. 실제값 · 예측값 · 기준모델 비교 그래프 ⭐ 결과물

범례의 이름을 클릭하면 선을 끄고 켤 수 있습니다. 실제(파란 선)와 가장 가깝게 붙어 다니는 선이 어느 것인지 보세요.

In [ ]:
fig = px.line(결과, x="월", y=["실제", "LSTM 예측", "전월값", "3개월평균"], markers=True,
              title="2025년 1월 ~ 2026년 6월 예금금리: 실제 vs 예측 (연 %)")
fig.show()

## 05-9. 누가 더 잘 맞혔나? — 평균 오차 비교

**평균 오차(MAE)** = 매달 예측이 실제에서 몇 %p 빗나갔는지의 평균입니다. **작을수록** 잘 맞힌 것입니다.

In [ ]:
오차표 = pd.DataFrame({
    "평균 오차(%p)": [(결과[모델] - 결과["실제"]).abs().mean() for 모델 in ["LSTM 예측", "전월값", "3개월평균"]]
}, index=["LSTM", "기준모델① 전월값", "기준모델② 3개월평균"]).round(3)
오차표.sort_values("평균 오차(%p)")

> 💡 **핵심 정리** — 18개월 전체로 보면 **"이번 달과 같다"는 단순한 기준모델(0.086%p)이 LSTM(0.104%p)보다 더 잘 맞혔습니다.**
> AI 모델이라고 항상 단순한 방법보다 낫지는 않습니다. 그래서 예측모델은 **반드시 기준모델과 비교해서** 평가합니다.

## 05-10. 언제 맞고, 언제 틀렸나?

시험 기간을 **하락이 이어지던 때**(2025년 1~8월)와 **방향이 바뀐 뒤**(2025년 9월~)로 나눠 봅니다.

In [ ]:
결과["구간"] = "① 하락 지속 (2025.1~8)"
결과.loc[결과["월"] >= "2025-09-01", "구간"] = "② 반등 이후 (2025.9~)"

for 모델 in ["LSTM 예측", "전월값"]:
    결과[모델 + " 오차"] = (결과[모델] - 결과["실제"]).abs()
결과.groupby("구간")[["LSTM 예측 오차", "전월값 오차"]].mean().round(3)

In [ ]:
결과.nlargest(3, "LSTM 예측 오차")[["월", "실제", "LSTM 예측", "LSTM 예측 오차"]]

> 💡 **핵심 정리** — LSTM은 금리가 **같은 방향으로 계속 내려갈 때는** 기준모델보다 잘 맞혔습니다(오차 0.050%p vs 0.090%p).
> 하지만 2025년 9월 이후 금리가 **다시 오르기 시작하자** "내려가던 흐름"을 계속 따라가다 크게 빗나갔습니다.
> 가장 크게 틀린 달은 2025년 11월(실제 2.81%, 예측 2.51%)입니다. **방향이 바뀌는 시점**이 예측모델이 가장 약한 곳입니다.

## 05-11. 다음 달(2026년 7월) 금리 예측

데이터의 마지막 12개월(2025년 7월 ~ 2026년 6월)을 넣어 **아직 데이터에 없는 다음 달**을 예측합니다.

In [ ]:
최근12개월 = 축소(금리[-W:]).reshape(1, W, 1)
다음달 = float(복원(model.predict(최근12개월, verbose=0)[0, 0]))
print("마지막 실제 금리 (2026년 6월):", 금리[-1], "%")
print("LSTM 예측 (2026년 7월)     :", round(다음달, 2), "%")

In [ ]:
최근 = pd.DataFrame({"월": 월.iloc[-24:], "금리": 금리[-24:]})
fig = px.line(최근, x="월", y="금리", markers=True, title="최근 2년 금리와 2026년 7월 LSTM 예측")
fig.add_scatter(x=[pd.Timestamp("2026-07-01")], y=[round(다음달, 2)], mode="markers+text",
                text=[f"예측 {다음달:.2f}%"], textposition="top center", marker_size=12, name="LSTM 예측")
fig.show()

> ⚠️ 이 숫자는 "다음 달 금리는 이렇게 된다"는 **약속이 아닙니다.** 과거 12개월의 모양만 보고 낸 **하나의 짐작**이며,
> 앞에서 확인했듯 방향이 바뀌는 시점에는 크게 틀릴 수 있습니다.

## 05-12. 🔧 가정 바꿔 보기 실험

"최근 3개월 금리가 실제보다 **0.3%p 높았다면**" 모델은 어떻게 예측할까요? `변화폭`을 바꿔 실행해 보세요. (예: `0.3`, `-0.3`, `0.5`)

In [ ]:
변화폭 = 0.3        # ← 바꿔 보세요 (단위: %p)

가정 = 금리[-W:].copy()
가정[-3:] = 가정[-3:] + 변화폭          # 최근 3개월만 바꾸기
가정예측 = float(복원(model.predict(축소(가정).reshape(1, W, 1), verbose=0)[0, 0]))
print("바꾼 최근 3개월:", 가정[-3:].round(2))
print("원래 예측:", round(다음달, 2), "% → 바꾼 뒤 예측:", round(가정예측, 2), "%")

> 💡 최근 숫자를 올리면 예측도 따라 올라갑니다. 모델은 **입력된 금리의 모양만** 보고 판단합니다.
> 기준금리 결정, 경기, 정책 발표처럼 금리를 실제로 움직이는 **바깥 정보는 전혀 모릅니다.**

## 05-13. [선택] 실제값과 맞춰 보기

한국은행 경제통계시스템(ecos.bok.or.kr)에서 같은 통계(121Y002, 항목 BEABAA2)의 **2026년 7월 실제값**을 찾아 아래에 넣고 비교해 보세요.

In [ ]:
실제_7월 = None      # ← ECOS에서 찾은 값을 넣으세요 (예: 3.05)

if 실제_7월 is not None:
    print("LSTM 예측:", round(다음달, 2), "% / 실제:", 실제_7월, "% / 오차:", round(abs(다음달 - 실제_7월), 2), "%p")
    print("전월값   :", 금리[-1], "% / 실제:", 실제_7월, "% / 오차:", round(abs(float(금리[-1]) - 실제_7월), 2), "%p")

## 05-14. 예측모델이 틀리는 이유와 금융업무 활용 시 주의사항 ⚠️

**왜 틀릴까?**
- **과거의 모양이 반복된다고 가정합니다.** 방향이 바뀌는 시점(2025년 하반기)처럼 처음 보는 흐름에는 약합니다.
- **금리만 보고 예측합니다.** 기준금리 결정, 물가, 경기, 정부 정책 같은 원인 정보가 들어 있지 않습니다.
- **데이터가 적습니다.** 월별 10년치는 126개, 학습 묶음은 96개뿐입니다. AI가 배우기에는 아주 적은 양입니다.
- **숫자 하나만 내놓습니다.** "3.01%"처럼 딱 떨어진 숫자는 정확해 보이지만, 실제로는 넓은 범위의 불확실성을 품고 있습니다.

**업무에 쓸 때 지킬 것**
1. 예측값은 **참고자료**로만 쓰고, 금리 결정·상품 설계의 단독 근거로 쓰지 않습니다.
2. 항상 **기준모델(전월값 등)과 비교**하고, 학습에 쓰지 않은 기간으로 검증한 결과만 믿습니다.
3. 정확도는 시기마다 달라집니다. **틀린 달을 따로 살펴보고** 왜 틀렸는지 기록합니다.
4. 보고할 때는 "예측 3.01%"가 아니라 **"단순 모델 기준 참고치이며, 방향 전환기에는 0.3%p 안팎 빗나간 적이 있음"** 처럼 한계를 함께 적습니다.

**💬 토의**: 우리 회사 업무에서 "다음 달 숫자"를 예측해 보고 싶은 지표는 무엇인가요? 그 지표를 움직이는 바깥 정보는 무엇일까요?

## ✅ 오늘 배운 것

| 하고 싶은 일 | 코드 |
|---|---|
| 0~1로 바꾸기 / 되돌리기 | `축소(x)`, `복원(z)` |
| 12개월 묶음 만들기 | `np.array([축소(금리[i - 12:i]) for i in 대상])` |
| 모델 모양 만들고 가중치 불러오기 | `keras.Sequential([...])`, `model.load_weights("파일")` |
| 예측하기 | `model.predict(X)` |
| 평균 오차(MAE) | `(예측 - 실제).abs().mean()` |

## 📝 복습 미션

> 아래 셀에서 **`# ←` 표시가 있는 줄만** 바꿔 실행하세요. 막히면 맨 아래 **정답 셀**을 실행해 결과를 확인합니다.

**미션 1.** 기준모델 ③ "최근 **2개월** 평균"을 만들어 평균 오차를 구해 보세요. LSTM보다 나은가요?
**이렇게 나오면 성공!** 0.117%p → LSTM(0.104%p)보다 조금 큼, 전월값(0.086%p)보다도 큼

**미션 2.** 05-12에서 `변화폭 = -0.3`으로 바꿔 실행해 보세요.
**이렇게 나오면 성공!** 바꾼 뒤 예측이 약 2.73%로 내려감

**오늘 꼭 가져갈 한 문장**
- [ ] AI 예측은 **기준모델과 비교**해서 평가하고, **방향이 바뀌는 시점에 약하다**는 것을 알고 쓴다

In [ ]:
# 미션 1 — 최근 n개월 평균 기준모델
n = 3        # ← 3을 2로
예측n = np.array([금리[i - n:i].mean() for i in 대상[시험]]).round(2)
print("평균 오차:", round(float(np.abs(예측n - 결과["실제"].to_numpy()).mean()), 3))

### 정답 (먼저 위 셀을 바꿔 실행해 본 뒤 확인하세요)

In [ ]:
# 미션 1 정답
결과["2개월평균"] = np.array([금리[i - 2:i].mean() for i in 대상[시험]]).round(2)
print(round(float((결과["2개월평균"] - 결과["실제"]).abs().mean()), 3))

---
### ▶ 1일 차 정리 · 2일 차 예고

| 예제 | 만든 것 |
|---|---|
| 01 | 저축은행 79개사 경영지표 비교표 |
| 02 · 03 | 고객군별 정기예금 가입률 분석표와 대시보드 |
| 04 · 05 | 금리 추세 그래프와 LSTM 예측 비교 |

**2일 차**에는 1일 차의 고객 데이터(`segment_rates.csv`, `bank_customers_prepared.csv`)를 그대로 이어 씁니다.
로컬 AI 에이전트(Ollama · Qwen 기반 FinanceAgent)로 고객군별 가입률을 Excel로 정리하고, 「정기예금 가입률 향상을 위한 고객관리 방안」 보고서를 자동으로 작성합니다.